# UNWRAPPING AI
## Zero to Transformer — Step 02: Mean Squared Error (MSE)

**Creator:** Nitin K. Sethi  
**Learning series:** Zero to Transformer — 12 Steps  
**Practical project:** House Price Prediction

---

## 1. Why do we need an error metric?

Imagine a model estimates a house price at $450,000 when the actual price is $500,000. The prediction is off by $50,000. Another model predicts $490,000, an error of only $10,000. We need a consistent way to measure and compare errors across many predictions.

**Mean Squared Error (MSE)** measures the average squared difference between actual values and predicted values. It is widely used to evaluate regression predictions and can also serve as a training loss.

## 2. Supervised learning: X and y

In supervised learning, a model learns from examples with known answers.

- **X (features):** information used to make a prediction, such as housing characteristics.
- **y (target):** the value we want to predict, such as a house-value estimate.
- **ŷ (y-hat):** the model's predicted value.

This notebook uses the California Housing dataset available through scikit-learn. Its target is expressed in units of $100,000, so MSE is in squared target units—not directly in dollars.

## 3. The MSE formula

\[
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
\]

Where `n` is the number of observations, `yᵢ` is the actual target, and `ŷᵢ` is the predicted target.

## 4. Why square the errors?

- **Positive and negative errors do not cancel out.** Errors of +10 and -10 both become 100 after squaring.
- **Large errors get a larger penalty.** An error of 10 contributes 100, while an error of 2 contributes only 4.
- **MSE is useful for optimization.** It is differentiable and commonly used with gradient-based methods.

MSE is sensitive to outliers, and its units are the square of the target's units. Root Mean Squared Error (RMSE), the square root of MSE, returns to the target's original units.

## 5. Why split into training and test data?

- **Training set:** supplies examples used to fit a model or calculate training-only baseline statistics.
- **Test set:** evaluates predictions on records held out from fitting.
- **80:20 split:** a common starting point, not a universal rule.

## 6. Our baseline: predict the training mean

Before building Linear Regression in Step 03, we need a simple reference point. Here, the baseline predicts the **mean target value from the training set** for every test observation. It does not use the input features.

In Step 03, we'll train a regression model and compare its test MSE against this baseline. Never use test targets to create predictions or fit the model.

## 7. What we'll do

1. Load the California Housing dataset.
2. Identify features (`X`) and target (`y`).
3. Split the data into training and test sets.
4. Create baseline predictions using the training-target mean.
5. Calculate MSE using scikit-learn.
6. Verify the result independently with NumPy.
7. Visualize actual values and baseline predictions.

## 8. Key takeaways

- MSE is the average squared prediction error.
- Squaring prevents positive and negative errors from cancelling.
- Large errors receive disproportionately greater penalties.
- Lower MSE is generally better when comparing predictions on the same target scale and evaluation data.
- This notebook establishes a baseline; it does **not** train a regression model yet.

---

**Unwrapping AI:** Understand the concept. Build the solution. Measure the results.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print("Unwrapping AI | Zero to Transformer | Step 02: MSE")

## Step 1 — Load the house-value dataset

The first run downloads the California Housing dataset through scikit-learn, so the Colab runtime needs internet access.

In [ ]:
# The first run downloads the California Housing dataset; Colab needs internet access.
housing = fetch_california_housing(as_frame=True)

X = housing.data
y = housing.target

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:")
print(list(X.columns))
print("\nFirst five rows:")
display(X.head())

print("\nTarget description:")
print(housing.target_names)
print("Target unit: $100,000")

## Step 2 — Split into training and test sets

We reserve 20% of the records for evaluation. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training share:", f"{len(X_train) / len(X):.1%}")
print("Testing share:", f"{len(X_test) / len(X):.1%}")
print("Mean target in training set:", round(y_train.mean(), 4))

## Step 3 — Create baseline predictions

The baseline predicts the training-set target mean for every test record. We do not use `y_test` to create predictions.

In [ ]:
# Predict the training-set target mean for every test record.
baseline_value = y_train.mean()
y_pred = np.full(shape=len(y_test), fill_value=baseline_value)

baseline_preview = pd.DataFrame({
    "Actual target": y_test.to_numpy()[:10],
    "Baseline prediction": y_pred[:10]
})
display(baseline_preview)
print("Number of test predictions:", len(y_pred))

## Step 4 — Calculate MSE with scikit-learn

MSE is the average of squared differences between actual test targets and baseline predictions.

In [ ]:
mse = mean_squared_error(y_test, y_pred)

print("Unwrapping AI | Baseline evaluation")
print(f"Mean Squared Error: {mse:.6f}")
print("Reminder: the target is measured in units of $100,000.")

## Step 5 — Verify MSE manually with NumPy

The two results should match up to floating-point precision.

In [ ]:
manual_mse = np.mean((y_test.to_numpy() - y_pred) ** 2)

print(f"scikit-learn MSE: {mse:.10f}")
print(f"NumPy MSE:        {manual_mse:.10f}")

assert np.isclose(mse, manual_mse)
assert mse >= 0
assert len(y_pred) == len(y_test)

print("\nPASS: Both MSE calculations agree.")
print("PASS: MSE is non-negative.")
print("PASS: There is one prediction per test record.")

## Step 6 — Visualize actual values and baseline predictions

The baseline predicts one constant value, so predictions appear as a horizontal band. The diagonal reference represents perfect predictions.

In [ ]:
actual = y_test.to_numpy()

plt.figure(figsize=(8, 5))
plt.scatter(actual, y_pred, alpha=0.25, label="Baseline predictions")

lower = min(actual.min(), y_pred.min())
upper = max(actual.max(), y_pred.max())
plt.plot([lower, upper], [lower, upper], linestyle="--",
         label="Perfect prediction")

plt.xlabel("Actual target (units of $100,000)")
plt.ylabel("Predicted target (units of $100,000)")
plt.title("Unwrapping AI | Step 02: MSE Baseline")
plt.legend()
plt.tight_layout()
plt.show()

## Final verification checklist

- [ ] Dataset loaded and feature/target shapes were displayed.
- [ ] Training and test sets were created using an 80:20 split.
- [ ] Baseline predictions used only the training-target mean.
- [ ] scikit-learn MSE and manual NumPy MSE match.
- [ ] Visualization rendered successfully.
- [ ] Notebook saved with outputs and uploaded to GitHub.

### What comes next?

In **Step 03 — Linear Regression**, we'll train a model using the input features, evaluate it on the same held-out test set, and compare its MSE with this baseline.

**Unwrapping AI — Learn the concept. Build the model. Understand the why.**